In [1]:
# Testing data.py
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.data import load_dataset, split_dataset

df = load_dataset(project_root /"data" / "IRMAS-TrainingData" / "IRMAS-TrainingData")

print(df.head())
print(df["label"].value_counts())
print("Total: ", len(df))

                                            filepath  label label_short
0  c:\Users\nimac\KTH\DT2470 Music Informatics\mu...  cello         cel
1  c:\Users\nimac\KTH\DT2470 Music Informatics\mu...  cello         cel
2  c:\Users\nimac\KTH\DT2470 Music Informatics\mu...  cello         cel
3  c:\Users\nimac\KTH\DT2470 Music Informatics\mu...  cello         cel
4  c:\Users\nimac\KTH\DT2470 Music Informatics\mu...  cello         cel
label
voice              778
electric_guitar    760
piano              721
organ              682
acoustic_guitar    637
saxophone          626
violin             580
trumpet            577
clarinet           505
flute              451
cello              388
Name: count, dtype: int64
Total:  6705


In [2]:
train_df, val_df, test_df = split_dataset(df)

print(len(train_df), len(val_df), len(test_df))

print(train_df["label"].value_counts(normalize=True))
print(val_df["label"].value_counts(normalize=True))
print(test_df["label"].value_counts(normalize=True))

print(len(train_df)+len(val_df)+len(test_df))
print(train_df["label"].nunique())
print(val_df["label"].nunique())
print(test_df["label"].nunique())

4693 1006 1006
label
voice              0.115917
electric_guitar    0.113360
piano              0.107607
organ              0.101641
acoustic_guitar    0.095035
saxophone          0.093330
violin             0.086512
trumpet            0.086086
clarinet           0.075218
flute              0.067334
cello              0.057959
Name: proportion, dtype: float64
label
voice              0.116302
electric_guitar    0.113320
piano              0.107356
organ              0.101392
acoustic_guitar    0.094433
saxophone          0.093439
violin             0.086481
trumpet            0.086481
clarinet           0.075547
flute              0.067594
cello              0.057654
Name: proportion, dtype: float64
label
voice              0.116302
electric_guitar    0.113320
piano              0.107356
organ              0.102386
acoustic_guitar    0.095427
saxophone          0.093439
violin             0.086481
trumpet            0.085487
clarinet           0.075547
flute              0.066600
cello

In [3]:
# Testing features.py

from src.features import extract_frame_features, extract_feature_vector, build_feature_matrix

sample_path = train_df.iloc[0]["filepath"]

features = extract_frame_features(sample_path)
print("MFCC:", features["mfcc"].shape)
print("Spectral centroid:", features["spectral_centroid"].shape)
print("ZCR:", features["zcr"].shape)

x1 = extract_feature_vector(sample_path, selected_feature="mfcc", pooling="mean")
x2 = extract_feature_vector(sample_path, selected_feature="combined", pooling="mean_std")

print(x1.shape)
print(x2.shape)

MFCC: (13, 259)
Spectral centroid: (1, 259)
ZCR: (1, 259)
(13,)
(30,)


In [4]:
x_train, y_train = build_feature_matrix(train_df, selected_feature="mfcc", pooling="mean")
x_val, y_val = build_feature_matrix(val_df, selected_feature="mfcc", pooling="mean")
x_test, y_test = build_feature_matrix(test_df, selected_feature="mfcc", pooling="mean")

print(x_train.shape, y_train.shape)
print(x_val.shape, y_val.shape)
print(x_test.shape, y_test.shape)

(4693, 13) (4693,)
(1006, 13) (1006,)
(1006, 13) (1006,)


In [5]:
# testing models

from src.models import fit_scaler, train_logistic_regression, train_svm

x_train_scaled, scaler = fit_scaler(x_train)
x_val_scaled = scaler.transform(x_val)

lr_model = train_logistic_regression(x_train_scaled, y_train)
svm_model = train_svm(x_train_scaled, y_train)

lr_pred = lr_model.predict(x_val_scaled)
svm_pred = svm_model.predict(x_val_scaled)


In [10]:
# testing evaluation
import importlib
import src.evaluation

importlib.reload(src.evaluation)

from src.evaluation import evaluate_model
print("Logistic Regression")
lr_results = evaluate_model(y_val, lr_pred)

print("\nSVM")
svm_results = evaluate_model(y_val, svm_pred)

Logistic Regression
Accuracy: 0.3907
Macro F1: 0.3860

                 precision    recall  f1-score   support

acoustic_guitar       0.37      0.43      0.40        95
          cello       0.52      0.43      0.47        58
       clarinet       0.51      0.39      0.44        76
electric_guitar       0.36      0.43      0.39       114
          flute       0.53      0.25      0.34        68
          organ       0.36      0.47      0.41       102
          piano       0.37      0.44      0.40       108
      saxophone       0.33      0.18      0.23        94
        trumpet       0.42      0.38      0.40        87
         violin       0.34      0.29      0.31        87
          voice       0.39      0.51      0.44       117

       accuracy                           0.39      1006
      macro avg       0.41      0.38      0.39      1006
   weighted avg       0.40      0.39      0.39      1006


SVM
Accuracy: 0.5010
Macro F1: 0.4921

                 precision    recall  f1-score 